In [1]:
import scanpy as sc
import anndata as ad
from matplotlib.gridspec import GridSpec
import matplotlib.pyplot as plt
from matplotlib_scalebar.scalebar import ScaleBar
from matplotlib.colors import ListedColormap, rgb2hex
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
import matplotlib.font_manager as fm
import numpy as np
import warnings
import pandas as pd
warnings.filterwarnings('ignore')
import numpy as np
from sklearn.metrics import jaccard_score
import seaborn as sns
import matplotlib.pyplot as plt
plt.rcParams['pdf.fonttype'] = 42 # ADOBE AI 字帖
import os
from matplotlib.font_manager import fontManager, FontProperties
import gc
from scipy.sparse import csr_matrix
fontManager.addfont('/data/work/Arial.ttf')

font = FontProperties(fname='/data/work/Arial.ttf')
font_name = font.get_name()
plt.rcParams['font.family'] = font_name

In [2]:
names = ['15_C03627F5_WT202403180043.h5ad',
         '17_C03627F6_WT202403270557.h5ad',
'19_D03657F1_WT202403110530.h5ad',
'21_D03657F2_WT202403110531.h5ad',
'22_B03606C4E6_WT202403310050.h5ad',
'23_B03609A4D6_WT202404150263.h5ad',
'27_B03610C1E3_WT202403310051.h5ad',
'31_B03619A1D3_WT202403310052.h5ad',
'35_B03619E4G6_WT202403310053.h5ad',
'39_A03589A1D4_WT202403310046.h5ad',
'43_A03590E1G4_WT202403310064.h5ad',
'47_A03593C1F3_WT202403310068.h5ad',
'51_B03605C2E5_WT202406020126.h5ad',
'55_B03613E3G6_WT202403310069.h5ad',
'59_B03612E4G6_WT202403310059.h5ad',
'63_B03606C1E3_WT202403310061.h5ad',
'67_A03595A1D3_WT202403310062.h5ad',
'71_A03595A4D6_WT202403310063.h5ad',
'76_D03656A5_WT202403280404.h5ad',
'81_D03657C6_WT202403110520.h5ad',
'85_B03611D2_WT202403110546.h5ad',
'90_A03592D3_WT202403110532.h5ad',
'95_B03602D1_WT202403110535.h5ad',
'100_B03609G1_WT202403280406.h5ad',
         'A03590A3D6_WT202407192652.h5ad', # gw13
         'A03588A1C2_WT202407161185.h5ad', # gw13
         'A03988A1C2_WT202407161208.h5ad', # gw13
         'A03994F1G2_WT2024071215067.h5ad',# gw13
         # 'A03591D4E5_WT2024071215074.h5ad',
         'A03587A5C6_WT2024071215080.h5ad', # gw10
         'B03607C4E6_WT2024071214941.h5ad', # gw12
         'B03618D3F6_WT202407152793.h5ad', # gw16
         'B04122A3F6_WT202407282762.h5ad', # gw18
]
colormap = {'Cebe_3': '#d590d7',
 'Cebe_4': '#e31fe9',
 'Cebe_7': '#ec2426',
 'Cebe_0': '#cc6c42',
 'Cebe_1': '#708365',
 'Cebe_11': '#8980c0',
 'Cebe_5': '#6e1f94',
 'Cebe_10': '#6fbf5e',
 'Cebe_6': '#3da672',
 'Cebe_2': '#902b32',
 'Cebe_9': '#cde114',
 'Cebe_8': '#bcf19a'}

In [3]:
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
import matplotlib.font_manager as fm

save_path = '/data/work/05.cluster/FuseMap/20251103/4_tracks/spatial_plot_20251104_1/'
h5ad_path = '/data/work/05.cluster/FuseMap/20251103/4_tracks/spatial_20251104_1/'

In [4]:
x_min, x_max, y_min, y_max = float('inf'), float('-inf'), float('inf'), float('-inf')
for name in names:
    try:
        adata_temp = sc.read_h5ad(f'{h5ad_path}{name}')
        adata_temp.obsm['align_spatial_2d'] = adata_temp.obsm['align_spatial_2d'] - adata_temp.obsm['align_spatial_2d'].max(axis = 0)
        x_min = min(x_min, adata_temp.obsm['align_spatial_2d'][:, 0].min())
        x_max = max(x_max, adata_temp.obsm['align_spatial_2d'][:, 0].max())
        y_min = min(y_min, adata_temp.obsm['align_spatial_2d'][:, 1].min())
        y_max = max(y_max, adata_temp.obsm['align_spatial_2d'][:, 1].max())
    except:
        print(name)

In [5]:
count = 0
fig = plt.figure(figsize=(64, 24))
gs = GridSpec(4, 8, figure=fig)
for name in names:
    try:
        adata_temp = sc.read_h5ad(f'{h5ad_path}{name}')
        means = adata_temp.obsm['align_spatial_2d'].max(axis = 0)
        adata_temp.obsm['align_spatial_2d'] = adata_temp.obsm['align_spatial_2d'] - means
        row = (count // 8) + 1
        col = count % 8  
        ax = fig.add_subplot(gs[row-1, col])

        sc.pl.embedding(
            adata_temp, basis="align_spatial_2d", color='ptime',
            show=False, s=1, title='', legend_loc=None, ax=ax, cmap = 'Reds',
        )

        X_grid, V_grid = adata_temp.uns['E_grid'],adata_temp.uns['V_grid']
        X_grid = X_grid.T - means
        X_grid = X_grid.T
        linewidth = 1
        density = 3
        arrow_color = 'k'
        arrow_size = 1
        arrow_style = '-|>'
        max_length = 4
        integration_direction = "both"
        lengths = np.sqrt((V_grid**2).sum(0))
        linewidth *= 2 * lengths / lengths[~np.isnan(lengths)].max()
        stream_kwargs = {
                "linewidth": linewidth,
                "density": density or 2,
                "zorder": 3,
                'arrow_color': 'k',
                "arrowsize": arrow_size or 1,
                "arrowstyle": arrow_style or "-|>",
                "maxlength": max_length or 4,
                "integration_direction": integration_direction or "both",
            }
        stream_kwargs["color"] = stream_kwargs.pop("arrow_color", "k")
        ax.streamplot(X_grid[0], X_grid[1], V_grid[0], V_grid[1], **stream_kwargs)
        ax.set_xlim(x_min, x_max)
        ax.set_ylim(y_min, y_max)

        ax.axis('off')
        ax.set_aspect('equal')
        if count == 0:
            scalebar = ScaleBar(0.0097, "mm", fixed_value=1, location = 'lower left', frameon = False,)
            ax.add_artist(scalebar)
        count += 1
    except:
        print(name)
plt.savefig(f'{save_path}ptime_stream_plot_all.png', bbox_inches = 'tight', dpi = 600)
plt.close()


In [6]:
count = 0
fig = plt.figure(figsize=(64, 24))
gs = GridSpec(4, 8, figure=fig)
for name in names:
    print(name)
    adata_temp = sc.read_h5ad(f'{h5ad_path}{name}')
    means = adata_temp.obsm['align_spatial_2d'].max(axis = 0)
    adata_temp.obsm['align_spatial_2d'] = adata_temp.obsm['align_spatial_2d'] - means
    row = (count // 8) + 1
    col = count % 8  
    ax = fig.add_subplot(gs[row-1, col])

    sc.pl.embedding(
        adata_temp, basis="align_spatial_2d", color='dmt_leiden_anno',
        show=False, s=1, title='', legend_loc=None, ax=ax, palette = colormap,
    )

    X_grid, V_grid = adata_temp.uns['E_grid'],adata_temp.uns['V_grid']
    X_grid = X_grid.T - means
    X_grid = X_grid.T
    linewidth = 1
    density = 3
    arrow_color = 'k'
    arrow_size = 1
    arrow_style = '-|>'
    max_length = 4
    integration_direction = "both"
    lengths = np.sqrt((V_grid**2).sum(0))
    linewidth *= 2 * lengths / lengths[~np.isnan(lengths)].max()
    stream_kwargs = {
            "linewidth": linewidth,
            "density": density or 2,
            "zorder": 3,
            'arrow_color': 'k',
            "arrowsize": arrow_size or 1,
            "arrowstyle": arrow_style or "-|>",
            "maxlength": max_length or 4,
            "integration_direction": integration_direction or "both",
        }
    stream_kwargs["color"] = stream_kwargs.pop("arrow_color", "k")
    ax.streamplot(X_grid[0], X_grid[1], V_grid[0], V_grid[1], **stream_kwargs)
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)

    ax.axis('off')
    ax.set_aspect('equal')
    if count == 0:
        scalebar = ScaleBar(0.0097, "mm", fixed_value=1, location = 'lower left', frameon = False,)
        ax.add_artist(scalebar)
    count += 1
plt.savefig(f'{save_path}region_stream_plot_all.png', bbox_inches = 'tight', dpi = 600)
plt.close()

15_C03627F5_WT202403180043.h5ad
17_C03627F6_WT202403270557.h5ad
19_D03657F1_WT202403110530.h5ad
21_D03657F2_WT202403110531.h5ad
22_B03606C4E6_WT202403310050.h5ad
23_B03609A4D6_WT202404150263.h5ad
27_B03610C1E3_WT202403310051.h5ad
31_B03619A1D3_WT202403310052.h5ad
35_B03619E4G6_WT202403310053.h5ad
39_A03589A1D4_WT202403310046.h5ad
43_A03590E1G4_WT202403310064.h5ad
47_A03593C1F3_WT202403310068.h5ad
51_B03605C2E5_WT202406020126.h5ad
55_B03613E3G6_WT202403310069.h5ad
59_B03612E4G6_WT202403310059.h5ad
63_B03606C1E3_WT202403310061.h5ad
67_A03595A1D3_WT202403310062.h5ad
71_A03595A4D6_WT202403310063.h5ad
76_D03656A5_WT202403280404.h5ad
81_D03657C6_WT202403110520.h5ad
85_B03611D2_WT202403110546.h5ad
90_A03592D3_WT202403110532.h5ad
95_B03602D1_WT202403110535.h5ad
100_B03609G1_WT202403280406.h5ad
A03590A3D6_WT202407192652.h5ad
A03588A1C2_WT202407161185.h5ad
A03988A1C2_WT202407161208.h5ad
A03994F1G2_WT2024071215067.h5ad
A03587A5C6_WT2024071215080.h5ad
B03607C4E6_WT2024071214941.h5ad
B03618D3F6_WT2

In [8]:
for celltype in colormap.keys():
    count = 0
    fig = plt.figure(figsize=(64, 24))
    gs = GridSpec(4, 8, figure=fig)
    for name in names:
        print(name)
        adata_temp = sc.read_h5ad(f'{h5ad_path}{name}')
        means = adata_temp.obsm['align_spatial_2d'].max(axis = 0)
        adata_temp.obsm['align_spatial_2d'] = adata_temp.obsm['align_spatial_2d'] - means
        row = (count // 8) + 1
        col = count % 8  
        ax = fig.add_subplot(gs[row-1, col])

        sc.pl.embedding(
            adata_temp, basis="align_spatial_2d", color='dmt_leiden_anno',groups = celltype,
            show=False, s=1, title='', legend_loc=None, ax=ax, palette = colormap,
        )

        X_grid, V_grid = adata_temp.uns['E_grid'],adata_temp.uns['V_grid']
        X_grid = X_grid.T - means
        X_grid = X_grid.T
        linewidth = 1
        density = 3
        arrow_color = 'k'
        arrow_size = 1
        arrow_style = '-|>'
        max_length = 4
        integration_direction = "both"
        lengths = np.sqrt((V_grid**2).sum(0))
        linewidth *= 2 * lengths / lengths[~np.isnan(lengths)].max()
        stream_kwargs = {
                "linewidth": linewidth,
                "density": density or 2,
                "zorder": 3,
                'arrow_color': 'k',
                "arrowsize": arrow_size or 1,
                "arrowstyle": arrow_style or "-|>",
                "maxlength": max_length or 4,
                "integration_direction": integration_direction or "both",
            }
        stream_kwargs["color"] = stream_kwargs.pop("arrow_color", "k")
        ax.streamplot(X_grid[0], X_grid[1], V_grid[0], V_grid[1], **stream_kwargs)
        ax.set_xlim(x_min, x_max)
        ax.set_ylim(y_min, y_max)

        ax.axis('off')
        ax.set_aspect('equal')
        if count == 0:
            scalebar = ScaleBar(0.0097, "mm", fixed_value=1, location = 'lower left', frameon = False,)
            ax.add_artist(scalebar)
        count += 1
    plt.savefig(f'{save_path}{celltype}_stream_plot_all.png', bbox_inches = 'tight', dpi = 600)
    plt.close()

15_C03627F5_WT202403180043.h5ad
17_C03627F6_WT202403270557.h5ad
19_D03657F1_WT202403110530.h5ad
21_D03657F2_WT202403110531.h5ad
22_B03606C4E6_WT202403310050.h5ad
23_B03609A4D6_WT202404150263.h5ad
27_B03610C1E3_WT202403310051.h5ad
31_B03619A1D3_WT202403310052.h5ad
35_B03619E4G6_WT202403310053.h5ad
39_A03589A1D4_WT202403310046.h5ad
43_A03590E1G4_WT202403310064.h5ad
47_A03593C1F3_WT202403310068.h5ad
51_B03605C2E5_WT202406020126.h5ad
55_B03613E3G6_WT202403310069.h5ad
59_B03612E4G6_WT202403310059.h5ad
63_B03606C1E3_WT202403310061.h5ad
67_A03595A1D3_WT202403310062.h5ad
71_A03595A4D6_WT202403310063.h5ad
76_D03656A5_WT202403280404.h5ad
81_D03657C6_WT202403110520.h5ad
85_B03611D2_WT202403110546.h5ad
90_A03592D3_WT202403110532.h5ad
95_B03602D1_WT202403110535.h5ad
100_B03609G1_WT202403280406.h5ad
A03590A3D6_WT202407192652.h5ad
A03588A1C2_WT202407161185.h5ad
A03988A1C2_WT202407161208.h5ad
A03994F1G2_WT2024071215067.h5ad
A03587A5C6_WT2024071215080.h5ad
B03607C4E6_WT2024071214941.h5ad
B03618D3F6_WT2

In [ ]:

stream_kwargs["color"] = stream_kwargs.pop("arrow_color", "k")
ax = sc.pl.embedding(adata, color = 'ptime', show = False, basis = 'align_spatial_2d_', cmap = 'Reds')
ax.streamplot(X_grid[0], X_grid[1], V_grid[0], V_grid[1], **stream_kwargs)


ax.axis('off')
ax.set_aspect('equal')
plt.show()

In [ ]:

stream_kwargs["color"] = stream_kwargs.pop("arrow_color", "k")
ax = sc.pl.embedding(adata, color = 'ptime', show = False, basis = 'align_spatial_2d_', cmap = 'Reds')
ax.streamplot(X_grid[0], X_grid[1], V_grid[0], V_grid[1], **stream_kwargs)


ax.axis('off')
ax.set_aspect('equal')
plt.show()

In [30]:
colormap = {'Cebe_0': '#a8fcff',
 'Cebe_3': '#9af59d',
 'Cebe_7': '#dd4cab',
 'Cebe_5': '#afa096',
 'Cebe_8': '#e0a8ab',
 'Cebe_6': '#f9ca38',
 'Cebe_2': '#a48a93',
 'Cebe_4': '#36a4a5',
 'Cebe_1': '#1db322'}
colormap = {'Cere_sc_08': '#cb30d3',
 'Cere_sc_15': '#11f791',
 'Cere_sc_06': '#680e4d',
 'Cere_sc_11': '#0a4140',
 'Cere_sc_13': '#8dce38',
 'Cere_sc_17': '#5a89c7',
 'Cere_sc_09': '#118ae2',
 'Cere_sc_19': '#8154e8',
 'Cere_sc_10': '#bc0267',
 'Cere_sc_02': '#3cde92',
 'Cere_sc_05': '#a437f8',
 'Cere_sc_12': '#01a4f2',
 'Cere_sc_16': '#67cc41',
 'Cere_sc_18': '#f3af8f',
 'Cere_sc_14': '#6b0c8e',
 'Cere_sc_07': '#5e7bbc',
 'Cere_sc_03': '#16dfdb',
 'Cere_sc_01': '#2ec289',
 'Cere_sc_04': '#5624e1'}